# 🔭 Fink/LSST Alert Browser

Interactive browser for Rubin/LSST alerts downloaded via the Fink broker.  
Inspired by the portal [https://lsst.fink-portal.org](https://lsst.fink-portal.org)

This notebook uses the **`fink_alert_lib`** library which provides:
- `FinkDataset` — dataset loader and index
- `plot_lightcurve_flux` / `plot_lightcurve_mag` — individual light curve plots
- `plot_cutouts` — Science / Template / Difference cutouts
- `plot_classifiers` — Fink classifier score bars
- `plot_alert_overview` — compact 5-panel portal-style view
- `plot_alert_detail` — full 2×3 detailed view
- `plot_tag_grid` — overview grid of all alerts for a tag
- `plot_tag_loop` — iterate and plot all alerts of a tag

**Column naming reminder (LSST DPDD schema):**
- Prefix `r:` → `diaSource` table field (≠ spectral band `r`)
- Prefix `f:` → Fink-computed field (classifiers, cross-matches)
- `r:band` ∈ {`u`, `g`, `r`, `i`, `z`, `y`} — the 6 Rubin spectral bands

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

%matplotlib inline
matplotlib.rcParams["figure.dpi"] = 120

# Import the Fink alert library
from fink_alert_lib import (
    FinkDataset,
    plot_lightcurve_flux,
    plot_lightcurve_mag,
    plot_cutouts,
    plot_classifiers,
    plot_alert_overview,
    plot_alert_detail,
    plot_tag_grid,
    plot_tag_loop,
    flux_to_mag,
    BAND_COLORS,
    DARK_BG,
)

print("✓ fink_alert_lib imported successfully")

In [ ]:
# Remove to run faster the notebook
# conda install -c conda-forge ipympl
import ipywidgets as widgets

%matplotlib widget

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────────────
DATASET_DIR = Path("fink_dataset")

ds = FinkDataset(DATASET_DIR)

print(f'Catalog      : {len(ds.catalog)} alerts, {ds.catalog["r:diaObjectId"].nunique()} unique objects')
print(f"Cutouts      : {len(ds.cutout_index)} .npy files")
print(f'Light curves : {len(list(ds.lc_dir.glob("*.parquet")))} .parquet files')
print(f"\nAvailable tags:")
print(ds.summary().to_string())

---
## 1 · Select a single alert

Set `TAG` and `INDEX` (or directly `OBJ_ID`) to choose the alert to display.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  SELECTION — edit TAG and INDEX to choose an alert             ║
# ╚══════════════════════════════════════════════════════════════════╝

# Available tags:
#   'extragalactic_new_candidate'       label=1
#   'extragalactic_lt20mag_candidate'   label=1
#   'sn_near_galaxy_candidate'          label=1
#   'in_tns'                            label=1
#   'hostless_candidate'                label=0

TAG = "sn_near_galaxy_candidate"  # ← choose a tag
INDEX = 0  # ← index within the tag (0 = first)

# List objects available for this tag
obj_table = ds.list_objects(TAG)
print(f"Objects available for tag '{TAG}' ({len(obj_table)} total):")
print(
    obj_table[
        [
            "r:diaObjectId",
            "r:band",
            "r:snr",
            "f:clf_snnSnVsOthers_score",
            "f:clf_earlySNIa_score",
            "f:xm_tns_fullname",
            "f:xm_legacydr8_zphot",
        ]
    ].to_string()
)

# Select the object
OBJ_ID = int(obj_table.loc[INDEX, "r:diaObjectId"])
print(f"\n→ Selected diaObjectId: {OBJ_ID}")
print(f"  Fink portal : https://lsst.fink-portal.org/{OBJ_ID}")

In [ ]:
# ── Print full metadata for the selected alert ────────────────────────────────
meta = ds.get_meta(OBJ_ID)
df_lc = ds.get_lightcurve(OBJ_ID)

print(f"diaObjectId  : {OBJ_ID}")
print(f'Tag          : {meta["fink_tag"]}  (label={meta["label"]})')
print(f'Coordinates  : RA={meta["r:ra"]:.6f}°   Dec={meta["r:dec"]:.6f}°')
print(f'Alert band   : {meta["r:band"]}   MJD={meta["r:midpointMjdTai"]:.4f} TAI')
print(f'psfFlux      : {meta["r:psfFlux"]:.1f} ± {meta["r:psfFluxErr"]:.1f} nJy   SNR={meta["r:snr"]:.1f}')
print(f'Extendedness : {meta["r:extendedness"]:.4f}  (0=point source, 1=extended)')
print(f'Reliability  : {meta["r:reliability"]:.4f}')
print()
print("Fink classifiers:")
print(f'  SNN (SN vs others) : {meta["f:clf_snnSnVsOthers_score"]:.4f}')
print(f'  Early SN Ia        : {meta["f:clf_earlySNIa_score"]:.4f}')
print(f'  CATS class/score   : {meta["f:clf_cats_class"]} / {meta["f:clf_cats_score"]:.4f}')
print()
print("Cross-matches:")
print(f'  TNS      : {meta.get("f:xm_tns_fullname", "—")}   type={meta.get("f:xm_tns_type", "—")}')
print(f'  SIMBAD   : {meta.get("f:xm_simbad_otype", "—")}')
print(
    f'  LegacyDR8: zphot={meta.get("f:xm_legacydr8_zphot", "—")}   pstar={meta.get("f:xm_legacydr8_pstar", "—")}'
)
print(f'  Mangrove : lum_dist={meta.get("f:xm_mangrove_lum_dist", "—")} Mpc')
print()
print(
    f'Light curve  : {len(df_lc)} measurements across bands {sorted(df_lc["r:band"].unique()) if not df_lc.empty else "—"}'
)
print(f"Fink portal  : https://lsst.fink-portal.org/{OBJ_ID}")

---
## 2 · Portal-style overview (5 panels)

In [ ]:
# Compact view: [flux LC] [mag LC] [Science] [Template] [Difference]
fig = plot_alert_overview(ds, OBJ_ID, save=False)
plt.show()

---
## 3 · Full detail view (2×3 grid)

In [ ]:
# Detailed view:
#   Row 0: flux LC | mag LC | classifier scores
#   Row 1: Science | Template | Difference cutouts
fig = plot_alert_detail(ds, OBJ_ID, save=False)
plt.show()

---
## 4 · Individual plots

In [ ]:
# ── Light curve in flux (nJy) ─────────────────────────────────────────────────
df_lc = ds.get_lightcurve(OBJ_ID)

fig, ax = plt.subplots(figsize=(9, 4), facecolor=DARK_BG)
plot_lightcurve_flux(df_lc, ax=ax)
ax.set_title(f"Flux light curve — diaObjectId {OBJ_ID}  |  {TAG}", color="#58a6ff", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ── Light curve in AB magnitude ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4), facecolor=DARK_BG)
plot_lightcurve_mag(df_lc, ax=ax)
ax.set_title(f"Magnitude light curve — diaObjectId {OBJ_ID}  |  {TAG}", color="#58a6ff", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cutouts only ─────────────────────────────────────────────────────────────
cutouts = ds.get_cutouts(OBJ_ID)
meta = ds.get_meta(OBJ_ID)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), facecolor=DARK_BG)
plot_cutouts(cutouts, band=meta["r:band"], axes=list(axes))
fig.suptitle(
    f'Cutouts — diaObjectId {OBJ_ID}  |  band {meta["r:band"]}  |  '
    f'MJD {meta["r:midpointMjdTai"]:.4f}  |  visit {meta["r:visit"]}',
    color="#e6edf3",
    fontsize=10,
    fontfamily="monospace",
)
plt.tight_layout()
plt.show()

In [ ]:
# ── Classifier scores only ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 3), facecolor=DARK_BG)
plot_classifiers(meta, ax=ax)
plt.tight_layout()
plt.show()

---
## 5 · Tag overview grid

In [ ]:
# Grid of all Science cutout thumbnails for the selected tag.
# The currently selected alert is highlighted with a yellow border.

fig = plot_tag_grid(
    ds,
    tag=TAG,
    ncols=5,
    highlight_id=OBJ_ID,  # yellow border on selected object
    save=False,
)
plt.show()

---
## 6 · Loop over alerts

Iterate over all (or `max_alerts`) alerts of a tag and display the chosen plot type.

Available `plot_type` values:
| Value | Description |
|---|---|
| `'overview'` | Compact 5-panel row (flux, mag, 3 cutouts) |
| `'detail'` | Full 2×3 grid (LCs + classifiers + cutouts) |
| `'lc_flux'` | Light curve in flux only |
| `'lc_mag'` | Light curve in magnitude only |
| `'cutouts'` | Cutouts only (1×3) |

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  LOOP PARAMETERS — edit as needed                              ║
# ╚══════════════════════════════════════════════════════════════════╝

LOOP_TAG = TAG  # tag to iterate over
LOOP_PLOT_TYPE = "overview"  # 'overview' | 'detail' | 'lc_flux' | 'lc_mag' | 'cutouts'
LOOP_MAX = 20  # max number of alerts to display (None = all)
LOOP_SAVE = False  # save each figure to disk

plot_tag_loop(
    ds,
    tag=LOOP_TAG,
    plot_type=LOOP_PLOT_TYPE,
    max_alerts=LOOP_MAX,
    save=LOOP_SAVE,
    close_after=True,  # close figures after display to save memory
)

---
## 7 · Cross-tag comparison

Display the detail view for the first alert of each available tag.

In [ ]:
# Show the overview plot for the first alert of every tag
for tag in ds.available_tags:
    oids = ds.get_object_ids(tag)
    if len(oids) == 0:
        continue
    obj_id = int(oids[0])
    print(f"\n{'='*60}")
    print(f"Tag: {tag}   →   diaObjectId: {obj_id}")
    print(f"{'='*60}")
    fig = plot_alert_overview(ds, obj_id, save=False)
    plt.show()
    plt.close(fig)

---
## 8 · Custom plot: flux + mag side by side for a specific object

Example of using the low-level functions directly to build a custom layout.

In [ ]:
# Custom layout: flux and mag light curves side by side, sharing the time axis
import matplotlib.pyplot as plt

df_lc = ds.get_lightcurve(OBJ_ID)
t0 = df_lc["r:midpointMjdTai"].min() if not df_lc.empty else 0

fig, (ax_flux, ax_mag) = plt.subplots(
    1,
    2,
    figsize=(14, 5),
    facecolor=DARK_BG,
    sharey=False,
)

plot_lightcurve_flux(df_lc, ax=ax_flux, t0=t0)
plot_lightcurve_mag(df_lc, ax=ax_mag, t0=t0)

fig.suptitle(
    f"diaObjectId {OBJ_ID}  —  tag: {TAG}  —  " f"portal: https://lsst.fink-portal.org/{OBJ_ID}",
    color="#e6edf3",
    fontsize=10,
    fontfamily="monospace",
)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()